# Hyperparameter Tuning

## Objective

Improve the trained models by finding better parameter values.

The objective of this notebook is to optimize the performance of each regression model by identifying the best hyperparameter combinations.

Hyperparameter optimization is performed using RandomizedSearchCV with 5-fold cross-validation.

The optimized models are then retrained using the best parameters and saved for final evaluation.



# Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import pandas as pd

from scipy.stats import randint
from scipy.stats import uniform

from sklearn.model_selection import RandomizedSearchCV

# Load Processed Data

# cycle and period model

In [2]:
X_train = joblib.load("../processed_data/Xc_train.pkl")
y_train = joblib.load("../processed_data/yc_train.pkl")

In [3]:
Xp_train = joblib.load("../processed_data/Xp_train.pkl")
yp_train = joblib.load("../processed_data/yp_train.pkl")

# Import Models

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

# Define Parameter Grids

# Decision Tree

In [5]:
dt_params = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# Random Forest

In [6]:
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

# KNN

In [7]:
knn_params = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "p": [1, 2]
}

# SVR

In [8]:
svr_params = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto"],
    "kernel": ["rbf", "linear"]
}

# Gradient Boosting

In [9]:
gb_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5]
}

# XGBoost

In [10]:
xgb_params = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3]
}

# Create Models

Linear Regression has no major hyperparameters, so it does not require tuning.

In [11]:
models = {
    "Decision Tree": (
        DecisionTreeRegressor(random_state=42),
        dt_params
    ),

    "Random Forest": (
        RandomForestRegressor(random_state=42),
        rf_params
    ),

    "KNN": (
        KNeighborsRegressor(),
        knn_params
    ),

    "SVR": (
        SVR(),
        svr_params
    ),

    "Gradient Boosting": (
        GradientBoostingRegressor(random_state=42),
        gb_params
    ),

    "XGBoost": (
        XGBRegressor(
            random_state=42,
            objective="reg:squarederror"
        ),
        xgb_params
    )
}

# Tune Cycle Models

In [12]:
best_cycle_models = {}
cycle_results = []

for name, (model, params) in models.items():

    print(f"Tuning {name}")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=20,
        cv=5,
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    best_cycle_models[name] = search.best_estimator_

    cycle_results.append({
        "Model": name,
        "Best Score": search.best_score_,
        "Best Parameters": search.best_params_
    })

    print("Done")

Tuning Decision Tree
Done
Tuning Random Forest
Done
Tuning KNN
Done
Tuning SVR
Done
Tuning Gradient Boosting
Done
Tuning XGBoost
Done


# Display Results

In [13]:
cycle_results = pd.DataFrame(cycle_results)

display(cycle_results)

,Model,Best Score,Best Parameters
0,Decision Tree,-2.825781,"{'min_samples_split': 10, 'min_samples_leaf': ..."
1,Random Forest,-2.855001,"{'n_estimators': 300, 'min_samples_split': 5, ..."
2,KNN,-3.182276,"{'weights': 'distance', 'p': 2, 'n_neighbors':..."
3,SVR,-2.852068,"{'kernel': 'linear', 'gamma': 'scale', 'C': 100}"
4,Gradient Boosting,-2.809396,"{'n_estimators': 100, 'max_depth': 3, 'learnin..."
5,XGBoost,-2.822525,"{'subsample': 1.0, 'n_estimators': 300, 'min_c..."


# Save Optimized Cycle Models

In [14]:
for name, model in best_cycle_models.items():

    filename = (
        "../models/"
        + name.replace(" ", "_")
        + "_cycle_best.pkl"
    )

    joblib.dump(model, filename)

# Tune Period Models

In [15]:
best_period_models = {}
period_results = []

for name, (model, params) in models.items():

    print(f"Tuning {name}")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=20,
        cv=5,
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=-1
    )

    search.fit(Xp_train, yp_train)

    best_period_models[name] = search.best_estimator_

    period_results.append({
        "Model": name,
        "Best Score": search.best_score_,
        "Best Parameters": search.best_params_
    })

    print(f"{name} Completed")

Tuning Decision Tree
Decision Tree Completed
Tuning Random Forest
Random Forest Completed
Tuning KNN
KNN Completed
Tuning SVR
SVR Completed
Tuning Gradient Boosting
Gradient Boosting Completed
Tuning XGBoost
XGBoost Completed


# View Results

In [16]:
period_results = pd.DataFrame(period_results)

display(period_results)

,Model,Best Score,Best Parameters
0,Decision Tree,-1.709842,"{'min_samples_split': 5, 'min_samples_leaf': 2..."
1,Random Forest,-1.710922,"{'n_estimators': 100, 'min_samples_split': 2, ..."
2,KNN,-1.751378,"{'weights': 'uniform', 'p': 1, 'n_neighbors': 11}"
3,SVR,-1.737289,"{'kernel': 'rbf', 'gamma': 'scale', 'C': 0.1}"
4,Gradient Boosting,-1.709520,"{'n_estimators': 100, 'max_depth': 3, 'learnin..."
5,XGBoost,-1.709857,"{'subsample': 1.0, 'n_estimators': 200, 'min_c..."


# Save Best Models

In [17]:
for name, model in best_period_models.items():

    filename = (
        "../models/"
        + name.replace(" ", "_")
        + "_period_best.pkl"
    )

    joblib.dump(model, filename)

# Save Results

In [18]:
period_results.to_csv(
    "../results/period_hyperparameter_results.csv",
    index=False
)

# Best parameters

In [19]:
search.best_params_

{'subsample': 1.0,
 'n_estimators': 200,
 'min_child_weight': 3,
 'max_depth': 3,
 'learning_rate': 0.01,
 'gamma': 0.3,
 'colsample_bytree': 1.0}